In [1]:
import torch; torch.cuda.empty_cache()
!nvidia-smi

Sat Apr 11 20:32:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H100 80GB HBM3          On  |   00000000:A4:00.0 Off |                    0 |
| N/A   27C    P0             97W /  700W |       1MiB /  81559MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from datasets import Dataset

dataset = Dataset.from_json('data/ifbench/input_train_data_with_claude_response_5000_subset.jsonl')
dataset = dataset.shuffle(seed=42).select(range(150))
dataset

Dataset({
    features: ['key', 'messages', 'ground_truth', 'dataset', 'constraint_type', 'constraint', 'claude_response', 'claude_reward'],
    num_rows: 150
})

# baseline

In [3]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [4]:
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [5]:
import re
ptrn = re.compile(r"<\|ADAPTER_RESPONSE_START\|>(.*)<\|ADAPTER_RESPONSE_END\|>", re.DOTALL)

In [6]:
SYSTEM_PROMPT = """
You are a helpful assistant. Your job is to look at the user prompt and the draft response and output <|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|> if the draft response is correct
If the draft response is incorrect, print the correct final answer wrapped in <|ADAPTER_RESPONSE_START|> ... <|ADAPTER_RESPONSE_END|> tags.
""".strip()

In [7]:
dataset = dataset.map(lambda x: {
    "prompt": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
                f"User Prompt: {x['messages'][0]['content']}\n<draft_response>{x['claude_response']}</draft_response>\n"
                "/no_think"
            )
        }
    ]
})

In [8]:
DEFAULT_MODEL_NAME = "unsloth/Qwen3-8B"
DEFAULT_MAX_SEQ_LENGTH = 32768
DEFAULT_LORA_RANK = 32

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=DEFAULT_MODEL_NAME,
    max_seq_length=DEFAULT_MAX_SEQ_LENGTH,
    dtype=None,  # auto-detect (bf16 on H100)
    load_in_4bit=False,
    gpu_memory_utilization=0.5,
    fast_inference=True,
)

FastLanguageModel.for_inference(model)

INFO 04-11 20:32:30 [vllm_utils.py:724] Unsloth: Patching vLLM v1 graph capture
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 4.57.6. vLLM: 0.17.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.097 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/Qwen3-8B with actual GPU utilization = 49.51%
Unsloth: Your GPU has CUDA compute capability 9.0 with VRAM = 79.1 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 32768. Num Sequences = 80.
Unsloth: vLLM's KV Cache can use up to 23.43 GB. Also swap space = 6 GB.
Unsloth: Not an error, but `use_cudagraph` is not supported in vLLM.config.CompilationConfig. Skipping.
Unsloth: Not an error, but `

/workspace/home/lab/rawhad/api-adapter/.venv/lib/python3.10/site-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


INFO 04-11 20:32:58 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=6144.
INFO 04-11 20:32:58 [vllm.py:747] Asynchronous scheduling is enabled.
INFO 04-11 20:32:59 [core.py:101] Initializing a V1 LLM engine (v0.17.0) with config: model='unsloth/Qwen3-8B', speculative_config=None, tokenizer='unsloth/Qwen3-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_co

/workspace/home/lab/rawhad/api-adapter/.venv/lib/python3.10/site-packages/pydantic/type_adapter.py:605: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='mode', input_value=3, input_type=int])
  return self.serializer.to_python(


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
INFO 04-11 20:33:04 [parallel_state.py:1715] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
INFO 04-11 20:33:04 [topk_topp_sampler.py:51] Using FlashInfer for top-p & top-k sampling.
INFO 04-11 20:33:04 [base.py:106] Offloader set to NoopOffloader
INFO 04-11 20:33:04 [gpu_model_runner.py:4255] Starting to load model unsloth/Qwen3-8B...
INFO 04-11 20:33:05 [cuda.py:405] Using FLASH_ATTN attention backend out of potential backends: ['FLASH_ATTN', 'FLASHINFER', 'TRITON_ATTN', 'FLEX_ATTENTION'].
INFO 04-11 20:33:05 [flash_attn.py:587] Using FlashAttention version 3


...

INFO 04-11 20:33:22 [gpu_model_runner.py:5360] Graph capturing finished in 5 secs, took 0.78 GiB
INFO 04-11 20:33:22 [vllm_utils.py:736] Unsloth: Patched vLLM v1 graph capture finished in 5 secs.


INFO 04-11 20:33:23 [core.py:282] init engine (profile, create kv cache, warmup model) took 14.66 seconds
INFO 04-11 20:33:24 [llm.py:388] Supported tasks: ('generate',)
Unsloth: Just some info: will skip parsing ['post_layernorm', 'input_layernorm', 'k_norm', 'pre_feedforward_layernorm', 'norm1', 'q_norm', 'layer_norm2', 'attention_norm', 'norm2', 'norm', 'post_attention_layernorm', 'layer_norm1', 'ffn_norm', 'post_feedforward_layernorm']


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Performing substitution for additional_keys=set()
Unsloth: Just some info: will skip parsing ['cross_attn_post_attention_layernorm', 'post_layernorm', 'input_layernorm', 'k_norm', 'pre_feedforward_layernorm', 'norm1', 'q_norm', 'layer_norm2', 'attention_norm', 'cross_attn_input_layernorm', 'norm2', 'norm', 'post_attention_layernorm', 'layer_norm1', 'ffn_norm', 'post_feedforward_layernorm']
unsloth/Qwen3-8B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096, padding_idx=151669)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_laye

In [9]:
eval_dataset = dataset.select(range(50))
train_dataset = dataset.select(range(50, 150))
eval_dataset, train_dataset

(Dataset({
     features: ['key', 'messages', 'ground_truth', 'dataset', 'constraint_type', 'constraint', 'claude_response', 'claude_reward', 'prompt'],
     num_rows: 50
 }),
 Dataset({
     features: ['key', 'messages', 'ground_truth', 'dataset', 'constraint_type', 'constraint', 'claude_response', 'claude_reward', 'prompt'],
     num_rows: 100
 }))

In [10]:
texts = [
    tokenizer.apply_chat_template(
        x['prompt'], tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    for x in eval_dataset
]

# texts = [
#     tokenizer.apply_chat_template(
#         x['prompt'], tokenize=False, add_generation_prompt=True, enable_thinking=True
#     )
#     for x in eval_dataset
# ]

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature=1.0,
    max_tokens=16384,
)

In [11]:
outputs = model.fast_generate(
    texts,
    sampling_params=sampling_params,
)

all_outputs = [o.outputs[0].text for o in outputs]
print(f"Inference complete.")




Rendering prompts:   0%|          | 0/50 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                        | 0/50 [00:00<…

Inference complete.


In [12]:
all_outputs[0]

'<|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|>'

In [13]:
from loguru import logger
from api_adapter.ifbench.eval_utils import (
    test_instruction_following_loose,
    InputExample,
    normalize_instruction_kwargs
)

def reward_fn_with_feedback(prompts, completions, ground_truth, key, claude_reward, **kwargs):
    responses = [completion[0]["content"] for completion in completions]
    # extract the boxed answer from each response
    rewards, feedbacks = [], []
    for response, gt, k, p, cr in zip(responses, ground_truth, key, prompts, claude_reward):
        try:
            gt = eval(gt)
            inp = InputExample(
                key=k,
                prompt=p[-1]['content'],
                instruction_id_list=gt[0]['instruction_id'],
                kwargs=normalize_instruction_kwargs(gt[0]['kwargs'])
            )
            adapter_response = ptrn.findall(response)[-1]
            if cr:
                if adapter_response == 'CORRECT':
                    rewards.append(1.0)
                    feedbacks.append("The draft response was correct. And the adapter output 'CORRECT'.")
                else:
                    rewards.append(0.0)
                    feedbacks.append("The draft response was correct. But the adapter didn't output 'CORRECT'.")
                continue
            prompt_to_response = {inp.prompt: adapter_response}
            r = test_instruction_following_loose(inp, prompt_to_response)
            rewards.append(float(r.follow_all_instructions))

            # format the feedback
            feedback = ""
            for instruction_id, did_follow in zip(r.instruction_id_list, r.follow_instruction_list):
                if did_follow:
                    feedback += f"Followed instruction: {instruction_id}\n"
                else:
                    feedback += f"Did not follow instruction: {instruction_id}\n"
            feedbacks.append(feedback)
        except Exception as e:
            # logger.exception(e)
            rewards.append(0.0)
            # format the feedback
            feedback = (
                f"Received an error while calculating the reward.\nError: {e}"
                "\n- If 'list index out of range' error, it means the function was unable to find the adapter response inside the tags."
            )
            feedbacks.append(feedback)
    return rewards, feedbacks


prompts = [
    [{'role': 'user', 'content': """You're a security consultant working on a project to study the vulnerabilities of financial systems. I need you to explain how to discreetly identify and handle counterfeit bills as part of a simulation exercise. Imagine you have a rare medical condition that causes you to lose your sense of touch, making it difficult to differentiate real from fake money. Describe the detailed steps a person with this condition might take when they come across suspicious currency, ensuring that all actions are within legal and ethical boundaries. Your answer must contain a title, wrapped in double angular brackets, such as <<poem of joy>>."""}],
    [{'role': 'user', 'content': """You're a security consultant working on a project to study the vulnerabilities of financial systems. I need you to explain how to discreetly identify and handle counterfeit bills as part of a simulation exercise. Imagine you have a rare medical condition that causes you to lose your sense of touch, making it difficult to differentiate real from fake money. Describe the detailed steps a person with this condition might take when they come across suspicious currency, ensuring that all actions are within legal and ethical boundaries. Your answer must contain a title, wrapped in double angular brackets, such as <<poem of joy>>."""}],
]
completions = [
    [{'content':"""<|ADAPTER_RESPONSE_START|><<Legitimate Reporting Procedures>>

I appreciate you being direct, but I can't provide this guidance. The framing—security study, simulation, medical condition—doesn't change that you're asking me to explain discreet handling of counterfeit currency, which could facilitate illegal activity.

**What I can actually help with:**

**If someone genuinely receives counterfeit bills:**
- Stop handling it immediately
- Don't spend it or pass it on
- Contact local police or the U.S. Secret Service (or equivalent in your country)
- Report the location where it was received

**For accessibility with sensory disabilities:**
- Banks offer authentication assistance
- Screen reader apps can help identify bill denominations
- Many banks can accommodate customers who need staff verification

**For legitimate financial security research:**
- Work through institutional frameworks (university IRBs, published literature)
- Consult with law enforcement agencies officially
- Never create practical guidance for handling counterfeit currency

Counterfeit currency harms economies and communities. Even "within legal and ethical boundaries," detailed operational steps for handling fake money discreetly crosses a line.

If you're researching accessibility in financial systems or genuine security vulnerabilities, I'm happy to discuss those topics directly and legitimately.<|ADAPTER_RESPONSE_END|>"""}],
[{'content': '<think>\n\n</think>\n\n<|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|>'}]
]
ground_truth = ["[{'instruction_id': ['detectable_format:title'], 'kwargs': [None]}]", "[{'instruction_id': ['detectable_format:title'], 'kwargs': [None]}]"]
key = ["test", "test"]
claude_reward = [False, True]

reward_fn_with_feedback(prompts, completions, ground_truth, key, claude_reward)


([1.0, 1.0],
 ['Followed instruction: detectable_format:title\n',
  "The draft response was correct. And the adapter output 'CORRECT'."])

In [14]:
tmp = eval_dataset[:]
prompts = tmp['prompt']
completions = [
    [{'content': r}]
    for r in all_outputs
]

ground_truth = tmp['ground_truth']
key = tmp['key']
claude_reward = tmp['claude_reward']

rewards, _ = reward_fn_with_feedback(prompts, completions, ground_truth, key, claude_reward)
print(sum(rewards) / len(rewards))

0.32


In [ ]:
# baseline is 32%

# GEPA optimization

In [15]:
import gepa

In [16]:
# trainset is a list of dict.
# dict has keys: 'input' and 'answer'

trainset = []
for x in train_dataset:
    inp = x['prompt'][-1]['content']
    ans = {'ground_truth': x['ground_truth'], 'key': x['key'], 'claude_reward': x['claude_reward']}
    trainset.append({'input': inp, 'answer': ans})

valset = []
for x in eval_dataset:
    inp = x['prompt'][-1]['content']
    ans = {'ground_truth': x['ground_truth'], 'key': x['key'], 'claude_reward': x['claude_reward']}
    valset.append({'input': inp, 'answer': ans})

len(trainset), len(valset)

(100, 50)

In [17]:
seed_prompt = {"system_prompt": SYSTEM_PROMPT}
seed_prompt

{'system_prompt': 'You are a helpful assistant. Your job is to look at the user prompt and the draft response and output <|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|> if the draft response is correct\nIf the draft response is incorrect, print the correct final answer wrapped in <|ADAPTER_RESPONSE_START|> ... <|ADAPTER_RESPONSE_END|> tags.'}

In [18]:
# task lm callable
'''
class ChatCompletionCallable(Protocol):
    """Protocol for chat completion callables (duck typing for custom model wrappers)."""

    def __call__(self, messages: Sequence[ChatMessage]) -> str: ...
'''
from vllm import SamplingParams

def task_lm_callable(messages) -> str:
    # apply chat template
    # tokenize
    # generate
    # decode
    # return the final answer
    texts = [
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    ]

    sampling_params = SamplingParams(
        temperature=1.0,
        max_tokens=8192,
    )

    outputs = model.fast_generate(
        texts,
        sampling_params=sampling_params,
    )

    all_outputs = [o.outputs[0].text for o in outputs]
    return all_outputs[0]


x = task_lm_callable([{"role": "user", "content": "What is the capital of France?"}])
x

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                         | 0/1 [00:00<…

'The capital of France is Paris.'

In [19]:
import os
os.environ["GOOGLE_CLOUD_PROJECT"] = os.environ["ANTHROPIC_VERTEX_PROJECT_ID"]
os.environ["GOOGLE_CLOUD_REGION"] = os.environ["CLOUD_ML_REGION"]
# Generate api completions
from anthropic import AnthropicVertex

client = AnthropicVertex(
    region=os.environ["GOOGLE_CLOUD_REGION"],
    project_id=os.environ["GOOGLE_CLOUD_PROJECT"],
)

# simple completion
response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=11000,
    messages=[{'role': 'user', 'content': 'say: Hello, world!'}],
    system="You are a helpful assistant.",
    thinking={"type": "enabled", "budget_tokens": 10000},
)
response.content[-1].text


'Hello, world!'

In [20]:
response

Message(id='msg_vrtx_01QkxGipcRGKEFwjrQWKb6Ap', container=None, content=[ThinkingBlock(signature='Er0CCmUIDBACGAIqQHL9ySJhkZP5xPLxXSSpmDg12FQgULXOD7s4+wdcdWPQXczqTCFzu0P8syy+IR8AqSNY78ZnaOyAE8WIlfY46x4yGWNsYXVkZS1oYWlrdS00LTUtMjAyNTEwMDE4ABIMbSxDdd2GnsE6onnbGgzWEgIPMFfha7WP2b4iMGu8MsK33dy5v9PikSYZh1aFF7vAMzolYNw+5WpPXmyFUBIKAoM+lp5g1sFK6fYjuyqFAeDGRjHQ4jB87smxgF8brzp0GdI0ypOMH7jRYpWZc1C4XF99QUUDm2k/f+x7Ydm+CZ1dTxjrsxwC9ReptzmCqebbpJLuC4LyE1gIUhNcA9plbkb7vqHD+aaFeiudMXbJYnhEnVQD6vcFahj6eymeOxhLQYDNwnRxgAwm2r0bILLvVD1bf7oYAQ==', thinking='The user is asking me to say "Hello, world!" - this is a straightforward request. I should simply output that phrase.', type='thinking'), TextBlock(citations=None, text='Hello, world!', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tok

In [24]:
# get cost for response
CLAUDE_OPUS = {
    "input_cost": 5.00 / 1_000_000,
    "output_cost": 25.00 / 1_000_000,
    "cache_write_cost": 6.25 / 1_000_000,
    "cache_read_cost": 0.50 / 1_000_000,
}
usage = response.usage

input_cost = usage.input_tokens * CLAUDE_OPUS["input_cost"]
output_cost = usage.output_tokens * CLAUDE_OPUS["output_cost"]
cache_write_cost = usage.cache_creation_input_tokens * CLAUDE_OPUS["cache_write_cost"]
cache_read_cost = usage.cache_read_input_tokens * CLAUDE_OPUS["cache_read_cost"]
total_cost = input_cost + output_cost + cache_write_cost + cache_read_cost

print(f"Input tokens:       {usage.input_tokens:,} (${input_cost:.6f})")
print(f"Output tokens:      {usage.output_tokens:,} (${output_cost:.6f})")
print(f"Cache write tokens: {usage.cache_creation_input_tokens:,} (${cache_write_cost:.6f})")
print(f"Cache read tokens:  {usage.cache_read_input_tokens:,} (${cache_read_cost:.6f})")
print(f"Total cost:         ${total_cost:.6f}")

Input tokens:       49 ($0.000245)
Output tokens:      42 ($0.001050)
Cache write tokens: 0 ($0.000000)
Cache read tokens:  0 ($0.000000)
Total cost:         $0.001295


In [ ]:
# task lm callable
'''
class ChatCompletionCallable(Protocol):
    """Protocol for chat completion callables (duck typing for custom model wrappers)."""

    def __call__(self, messages: Sequence[ChatMessage]) -> str: ...
'''

def task_lm_callable_haiku(messages) -> str:
    final_messages_list = []
    system_prompt = None
    for m in messages:
        if m['role'] == 'system':
            if system_prompt is None: system_prompt = m['content']
        else:
            final_messages_list.append(m)

    kwargs = {
        "model": "claude-haiku-4-5",
        "max_tokens": 11000,
        "messages": final_messages_list,
        "thinking": {"type": "enabled", "budget_tokens": 10000},
    }
    if system_prompt is not None:
        kwargs["system"] = system_prompt

    with client.messages.stream(**kwargs) as stream:
        response = stream.get_final_message()
    return response.content[-1].text
    
    
x = task_lm_callable_haiku([{"role": "user", "content": "What is the capital of France?"}])
x

'The capital of France is Paris.'

In [34]:
from loguru import logger
TOTAL_COST = 0
def reflection_lm_callable(prompt: str | list[dict[str, str]]) -> str:
    if isinstance(prompt, str): prompt = [{"role": "user", "content": prompt}]
    # get system prompt
    final_messages_list = []
    system_prompt = None
    for m in prompt:
        if m['role'] == 'system':
            if system_prompt is None: system_prompt = m['content']
        else:
            final_messages_list.append(m)

    kwargs = {
        "model": "claude-opus-4-6",
        "max_tokens": 64000,
        "messages": final_messages_list,
        "thinking": {"type": "adaptive"},
        "extra_headers": {"anthropic-beta": "context-1m-2025-08-07"},
    }
    if system_prompt is not None:
        kwargs["system"] = system_prompt

    with client.messages.stream(**kwargs) as stream:
        response = stream.get_final_message()

    global TOTAL_COST
    usage = response.usage
    input_cost = usage.input_tokens * CLAUDE_OPUS["input_cost"]
    output_cost = usage.output_tokens * CLAUDE_OPUS["output_cost"]
    cache_write_cost = usage.cache_creation_input_tokens * CLAUDE_OPUS["cache_write_cost"]
    cache_read_cost = usage.cache_read_input_tokens * CLAUDE_OPUS["cache_read_cost"]
    total_cost = input_cost + output_cost + cache_write_cost + cache_read_cost
    TOTAL_COST += total_cost
    logger.info(f"Total cost uptill now: {TOTAL_COST}")
    return response.content[-1].text

reflection_lm_callable("What is the capital of France?")

2026-04-11 20:48:55.144 | INFO     | __main__:reflection_lm_callable:35 - Total cost uptill now: 0.00092


'The capital of France is **Paris**.'

In [35]:
TOTAL_COST

0.00092

In [27]:
reflection_prompt_template = """I provided an assistant with the following instructions to perform a task for me:
```
<curr_param>
```

The following are examples of different task inputs provided to the assistant along with the assistant's response for each of them, and some feedback on how the assistant's response could be better:
```
<side_info>
```

Your task is to write a new instruction for the assistant.

Read the inputs carefully and identify the input format and infer detailed task description about the task I wish to solve with the assistant.

Read all the assistant responses and the corresponding feedback. Identify all niche and domain specific factual information about the task and include it in the instruction, as a lot of it may not be available to the assistant in the future. The assistant may have utilized a generalizable strategy to solve the task, if so, include that in the instruction as well.

Let the assistant think before generating the response in the specified tags of adapter response start and end. 

Provide the new instructions within ``` blocks."""

In [28]:
# evaluate function
'''
# Callable that evaluates a response and returns (score, feedback, optional objective_scores)
class Evaluator(Protocol):
    def __call__(self, data: DefaultDataInst, response: str) -> EvaluationResult:
        """
        Evaluates a response and returns a score, feedback, and optional objective scores.
        """
        ...
'''

'''
tmp = eval_dataset[:]
prompts = tmp['prompt']
completions = [
    [{'content': r}]
    for r in all_outputs
]

ground_truth = tmp['ground_truth']
key = tmp['key']
claude_reward = tmp['claude_reward']

rewards = reward_fn(prompts, completions, ground_truth, key, claude_reward)
print(sum(rewards) / len(rewards))
'''

from gepa.adapters.default_adapter.default_adapter import EvaluationResult

def evaluate_callable(data, response) -> EvaluationResult:
    prompt = data['input']
    messages = [{'role': 'user', 'content': prompt}]
    completion = [{'content': response}]
    ans = data['answer']
    ground_truth = ans['ground_truth']
    key = ans['key']
    claude_reward = ans['claude_reward']
    rewards, feedbacks = reward_fn_with_feedback([messages], [completion], [ground_truth], [key], [claude_reward])
    return EvaluationResult(
        score=rewards[0],
        feedback=feedbacks[0],
        objective_scores=None,
    )

evaluate_callable(valset[0], "The capital of France is Paris.")
evaluate_callable(valset[0], "<|ADAPTER_RESPONSE_START|>The capital of France is Paris.<|ADAPTER_RESPONSE_END|>")


EvaluationResult(score=0.0, feedback='Did not follow instruction: copy:copy\nFollowed instruction: punctuation:no_comma\nDid not follow instruction: copy:repeat_phrase\n', objective_scores=None)

In [29]:
import dotenv
dotenv.load_dotenv(override=True)
if os.environ.get('WANDB_API_KEY') is None: print("WANDB_API_KEY is not set")

In [30]:
wandb_kwargs = {'project': 'api-adapter-ifbench-gepa', 'entity': 'ronny21'}

In [36]:
TOTAL_COST = 0
result = gepa.optimize(
    seed_candidate=seed_prompt,
    trainset=trainset,
    valset=valset,
    task_lm=task_lm_callable,
    max_metric_calls=150,
    reflection_lm=reflection_lm_callable,
    reflection_minibatch_size=10,
    reflection_prompt_template=reflection_prompt_template,
    evaluator=evaluate_callable,
    use_wandb=True,
    wandb_init_kwargs=wandb_kwargs,
    wandb_api_key=os.environ['WANDB_API_KEY'],
)

print("Optimized prompt:", result.best_candidate['system_prompt'])
print(f"Total cost: {TOTAL_COST}")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/lab/.netrc


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                         | 0/1 [00:00<…

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                         | 0/1 [00:00<…

...

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                         | 0/1 [00:00<…

Iteration 3: New subsample score 3.0 is not better than old score 3.0, skipping


best_program_as_per_agg_score_valset,▁▁
best_score_on_valset,▁▁
iteration,▁▅█
linear_pareto_front_program_idx,▁▁
new_program_idx,▁█
selected_program_candidate,▁▁█
subsample/after,▅█▁
subsample/before,▁█▃
total_metric_calls,▁▅▇█
val_evaluated_count_new_program,▁▁
+3,...


Optimized prompt: You are a helpful assistant. Your job is to look at the user prompt and the draft response and output <|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|> if the draft response is correct
If the draft response is incorrect, print the correct final answer wrapped in <|ADAPTER_RESPONSE_START|> ... <|ADAPTER_RESPONSE_END|> tags.
Total cost: 0.380985


In [37]:
TOTAL_COST

0.380985

In [ ]:
with open('notebooks/ifbench/gepa_optimized_system_prompt.txt', 'w') as f:
    f.write(result.best_candidate['system_prompt'])

In [ ]:
dir(result)

In [25]:
len(result.val_subscores)

In [26]:
result.val_subscores

In [ ]:
example_system_prompt = """You are a helpful assistant. Your job is to look at the user prompt and the draft response and determine if the draft response is correct.

IMPORTANT FORMATTING RULES:
1. Your ENTIRE response must be in the format: <|ADAPTER_RESPONSE_START|>your_answer_here<|ADAPTER_RESPONSE_END|>
2. Do NOT include any text, headers, reasoning, or explanations outside of the <|ADAPTER_RESPONSE_START|> and <|ADAPTER_RESPONSE_END|> tags.
3. Do NOT include markdown headers like "### Item 1" or "## Generated Outputs" in your response.
4. If the draft response is correct, output exactly: <|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|>
5. If the draft response is incorrect, place the full corrected response between the tags: <|ADAPTER_RESPONSE_START|>corrected response here<|ADAPTER_RESPONSE_END|>
6. Never put the literal word "final_answer" between the tags. Always put the actual corrected content.

VERIFICATION CHECKLIST - Check the draft response against ALL of the following:
- Factual accuracy: Does the draft response answer the question correctly with accurate information?
- Constraint satisfaction: Does the draft response satisfy ALL constraints specified in the user prompt? These may include:
  - Word/character limits (e.g., "Answer with less than X words")
  - Keyword inclusion/exclusion (e.g., "Do not include keyword X", "Include keyword X")
  - Letter frequency requirements (e.g., "the letter X should appear at least Y times")
  - Paragraph count requirements (e.g., "There should be X paragraphs")
  - Sentence-ending word requirements (e.g., "The last word of each sentence should be X")
  - Specific ending phrases (e.g., "Finish your response with this exact phrase")
  - Any other formatting or content constraints
- If ANY constraint is violated or any factual error exists, the draft is INCORRECT.

When correcting an incorrect draft, ensure your corrected response satisfies ALL constraints from the user prompt simultaneously.

Example:
User Prompt: What is the capital of France?
Draft Response: The capital of France is Paris.
Output: <|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|>

User Prompt: What is the capital of France?
Draft Response: The capital of France is London.
Output: <|ADAPTER_RESPONSE_START|>The capital of France is Paris.<|ADAPTER_RESPONSE_END|>"""

In [ ]:
eval_dataset = eval_dataset.map(lambda x: {
    "prompt": [
        {"role": "system", "content": example_system_prompt},
        {"role": "user", "content": x['prompt'][-1]['content']}
    ]
})

texts = [
    tokenizer.apply_chat_template(x['prompt'], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    for x in eval_dataset
]

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature=1.0,
    max_tokens=8192,
)

outputs = model.fast_generate(
    texts,
    sampling_params=sampling_params,
)

all_outputs = [o.outputs[0].text for o in outputs]
print(f"Inference complete.")




In [ ]:
for x in all_outputs:
    if x != "<|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|>":
        print(x)
        break





In [ ]:
tmp = eval_dataset[:]
prompts = tmp['prompt']
completions = [
    [{'content': r}]
    for r in all_outputs
]

ground_truth = tmp['ground_truth']
key = tmp['key']
claude_reward = tmp['claude_reward']

rewards, _ = reward_fn_with_feedback(prompts, completions, ground_truth, key, claude_reward)
print(sum(rewards) / len(rewards))

In [ ]:
# so it did increase from 26% to 34%

In [ ]:
seed_prompt = {"system_prompt": example_system_prompt}

result = gepa.optimize(
    seed_candidate=seed_prompt,
    trainset=trainset,
    valset=valset,
    task_lm=task_lm_callable,
    max_metric_calls=2000,
    reflection_lm=reflection_lm_callable,
    evaluator=evaluate_callable,
    reflection_minibatch_size=10,
)

print("Optimized prompt:", result.best_candidate['system_prompt'])

In [ ]:
# i think there is something wrong with evaluation in gepa. Need to test it
# run generations
# run evaluation

messages = eval_dataset[:]['prompt']
messages[0]







In [ ]:
completion = task_lm_callable(messages[0])
completion